# Hybrid Robotic Vision: Notebook 2 — YOLO ROI Manifest Generation

This notebook is the rewritten Notebook 2 for the hybrid visual telemetry experiment.

Its job is to simulate the lightweight onboard perception stage:

1. Load one original source video  
2. Load the corresponding encoded base-video stream  
3. Run a lightweight YOLO detector on the encoded/base video  
4. Generate ROI candidates every `N` frames  
5. Apply simple trigger and selection rules  
6. Map ROI coordinates from encoded-video geometry to original-video geometry  
7. Save a clean ROI manifest for later use

## Important design choice

This notebook does **not** save crops.

Instead, it saves only:
- detection results
- ROI candidates
- mapped ROI coordinates
- frame-index mappings
- metadata needed to reconstruct crops later

That keeps Notebook 2 focused on ROI proposal, while Notebook 3 will later materialize the real still-image side channel.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Define project paths and select the sequence

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/hybrid_robotic_vision")
DATA = ROOT / "data" / "uavdt"
VIDEOS = DATA / "videos"
RUNS = ROOT / "runs"

SEQUENCE_NAME = "M0101"   # <-- EDIT THIS
REGIME = "low"            # <-- choose: "low" or "moderate"

ORIGINAL_VIDEO = VIDEOS / f"{SEQUENCE_NAME}.mp4"

if REGIME == "low":
    ENCODED_VIDEO_DIR = RUNS / "uavdt_low" / "video_only" / SEQUENCE_NAME
elif REGIME == "moderate":
    ENCODED_VIDEO_DIR = RUNS / "uavdt_moderate" / "video_only" / SEQUENCE_NAME
else:
    raise ValueError("REGIME must be 'low' or 'moderate'")

ROI_RUN_DIR = RUNS / "yolo_roi_manifest" / SEQUENCE_NAME / REGIME
DET_DIR = ROI_RUN_DIR / "detections"
ROI_DIR = ROI_RUN_DIR / "roi_candidates"
MANIFEST_DIR = ROI_RUN_DIR / "manifests"

for p in [DET_DIR, ROI_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Original video:", ORIGINAL_VIDEO)
print("Encoded video dir:", ENCODED_VIDEO_DIR)
print("Output dir:", ROI_RUN_DIR)

## 3. Find the encoded video

In [ ]:
encoded_video_candidates = [p for p in ENCODED_VIDEO_DIR.iterdir() if p.suffix.lower() in [".mp4", ".mkv", ".mov"]]
if len(encoded_video_candidates) == 0:
    raise FileNotFoundError(f"No encoded video found in {ENCODED_VIDEO_DIR}")

ENCODED_VIDEO = encoded_video_candidates[0]
print("Using encoded video:")
print(ENCODED_VIDEO)

## 4. Install packages and tools

In [ ]:
!pip install -q ultralytics opencv-python-headless pandas tqdm matplotlib
!apt-get update -qq
!apt-get install -y ffmpeg

## 5. Imports

In [ ]:
import cv2
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

## 6. Inspect video properties

In [ ]:
def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_path}")
    info = {
        "path": str(video_path),
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": float(cap.get(cv2.CAP_PROP_FPS)),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    cap.release()
    return info

orig_info = get_video_info(ORIGINAL_VIDEO)
enc_info = get_video_info(ENCODED_VIDEO)

print("Original video info:")
print(json.dumps(orig_info, indent=2))

print("\nEncoded video info:")
print(json.dumps(enc_info, indent=2))

## 7. Load YOLO detector

In [ ]:
from ultralytics import YOLO

YOLO_MODEL_NAME = "yolov8n.pt"  # change to yolov8s.pt if desired
model = YOLO(YOLO_MODEL_NAME)

print("Loaded model:", YOLO_MODEL_NAME)

## 8. Detection sweep settings

In [ ]:
START_FRAME = 0
MAX_FRAMES = 300        # set to None for all frames
FRAME_STRIDE = 5
YOLO_CONF = 0.25
YOLO_IOU = 0.45

print("Detection settings:")
print("START_FRAME =", START_FRAME)
print("MAX_FRAMES =", MAX_FRAMES)
print("FRAME_STRIDE =", FRAME_STRIDE)
print("YOLO_CONF =", YOLO_CONF)
print("YOLO_IOU =", YOLO_IOU)

## 9. Run YOLO on the encoded/base video

In [ ]:
detections = []

cap = cv2.VideoCapture(str(ENCODED_VIDEO))
if not cap.isOpened():
    raise RuntimeError(f"Could not open encoded video: {ENCODED_VIDEO}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
end_frame = total_frames if MAX_FRAMES is None else min(total_frames, START_FRAME + MAX_FRAMES)

frame_idx = 0

with tqdm(total=end_frame, desc="YOLO on encoded video") as pbar:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx >= end_frame:
            break

        if frame_idx >= START_FRAME and ((frame_idx - START_FRAME) % FRAME_STRIDE == 0):
            results = model.predict(frame, conf=YOLO_CONF, iou=YOLO_IOU, verbose=False)
            r = results[0]

            if r.boxes is not None and len(r.boxes) > 0:
                xyxy = r.boxes.xyxy.cpu().numpy()
                confs = r.boxes.conf.cpu().numpy()
                clss = r.boxes.cls.cpu().numpy().astype(int)

                for i in range(len(xyxy)):
                    x1, y1, x2, y2 = xyxy[i].tolist()
                    cls_id = int(clss[i])
                    cls_name = model.names.get(cls_id, str(cls_id))
                    detections.append({
                        "enc_frame_idx": frame_idx,
                        "enc_x1": float(x1),
                        "enc_y1": float(y1),
                        "enc_x2": float(x2),
                        "enc_y2": float(y2),
                        "conf": float(confs[i]),
                        "cls_id": cls_id,
                        "cls_name": cls_name,
                    })

        frame_idx += 1
        pbar.update(1)

cap.release()

detections_df = pd.DataFrame(detections)
detections_csv = DET_DIR / "detections.csv"
detections_df.to_csv(detections_csv, index=False)

print(f"Saved {len(detections_df)} detections to:")
print(detections_csv)
detections_df.head()

## 10. Quick detection summary

In [ ]:
if len(detections_df) == 0:
    print("No detections found. Consider lowering YOLO_CONF or using a stronger model.")
else:
    display(detections_df["cls_name"].value_counts().to_frame("count"))

## 11. ROI trigger and selection rules

In [ ]:
MIN_CONF_FOR_KEEP = 0.20
MAX_ROIS_PER_FRAME = 3
SMALL_OBJECT_AREA_FRAC = 0.015  # 1.5% of encoded frame area
PADDING_FRAC = 0.15

ENC_FRAME_AREA = enc_info["width"] * enc_info["height"]

print("ROI selection parameters:")
print("MIN_CONF_FOR_KEEP =", MIN_CONF_FOR_KEEP)
print("MAX_ROIS_PER_FRAME =", MAX_ROIS_PER_FRAME)
print("SMALL_OBJECT_AREA_FRAC =", SMALL_OBJECT_AREA_FRAC)
print("PADDING_FRAC =", PADDING_FRAC)

In [ ]:
def compute_area_frac(row, frame_area):
    w = max(0.0, row["enc_x2"] - row["enc_x1"])
    h = max(0.0, row["enc_y2"] - row["enc_y1"])
    return (w * h) / frame_area if frame_area > 0 else 0.0

det_aug = detections_df.copy()
if len(det_aug) > 0:
    det_aug["area_frac"] = det_aug.apply(lambda r: compute_area_frac(r, ENC_FRAME_AREA), axis=1)
    det_aug["is_small"] = det_aug["area_frac"] < SMALL_OBJECT_AREA_FRAC
    det_aug["low_conf_trigger"] = det_aug["conf"] < 0.50
    det_aug["size_trigger"] = det_aug["is_small"]
    det_aug["priority_score"] = (
        det_aug["is_small"].astype(float) * 2.0 +
        (1.0 - det_aug["conf"]) * 1.0
    )
else:
    det_aug["area_frac"] = []
    det_aug["is_small"] = []
    det_aug["low_conf_trigger"] = []
    det_aug["size_trigger"] = []
    det_aug["priority_score"] = []

roi_candidates = []
for enc_frame_idx, group in det_aug.groupby("enc_frame_idx"):
    group = group[group["conf"] >= MIN_CONF_FOR_KEEP].copy()
    if len(group) == 0:
        continue

    selected = group.sort_values("priority_score", ascending=False).head(MAX_ROIS_PER_FRAME).copy()
    selected["roi_reason"] = np.where(selected["is_small"], "small_object", "low_conf_or_topk")
    roi_candidates.append(selected)

if len(roi_candidates) > 0:
    roi_df = pd.concat(roi_candidates, ignore_index=True)
else:
    roi_df = pd.DataFrame(columns=list(det_aug.columns) + ["roi_reason"])

roi_csv = ROI_DIR / "roi_candidates.csv"
roi_df.to_csv(roi_csv, index=False)

print(f"Saved {len(roi_df)} ROI candidates to:")
print(roi_csv)
roi_df.head()

## 12. Map frame indices and box coordinates into original-video space

In [ ]:
def map_encoded_frame_to_original(enc_frame_idx, enc_fps, orig_fps, orig_frame_count=None):
    t = enc_frame_idx / enc_fps
    orig_idx = int(round(t * orig_fps))
    if orig_frame_count is not None:
        orig_idx = max(0, min(orig_idx, orig_frame_count - 1))
    return orig_idx

def scale_box_between_frames(x1, y1, x2, y2, src_w, src_h, dst_w, dst_h):
    sx = dst_w / src_w
    sy = dst_h / src_h
    return (
        x1 * sx,
        y1 * sy,
        x2 * sx,
        y2 * sy,
    )

orig_fps = orig_info["fps"]
enc_fps = enc_info["fps"]
orig_frame_count = orig_info["frame_count"]
orig_w, orig_h = orig_info["width"], orig_info["height"]
enc_w, enc_h = enc_info["width"], enc_info["height"]

roi_manifest_rows = []

for idx, row in roi_df.iterrows():
    enc_frame_idx = int(row["enc_frame_idx"])
    orig_frame_idx = map_encoded_frame_to_original(
        enc_frame_idx,
        enc_fps,
        orig_fps,
        orig_frame_count=orig_frame_count
    )

    ox1, oy1, ox2, oy2 = scale_box_between_frames(
        float(row["enc_x1"]), float(row["enc_y1"]),
        float(row["enc_x2"]), float(row["enc_y2"]),
        enc_w, enc_h,
        orig_w, orig_h
    )

    roi_manifest_rows.append({
        "sequence_name": SEQUENCE_NAME,
        "regime": REGIME,
        "det_idx": int(idx),
        "enc_frame_idx": enc_frame_idx,
        "orig_frame_idx": int(orig_frame_idx),
        "cls_id": int(row["cls_id"]),
        "cls_name": row["cls_name"],
        "conf": float(row["conf"]),
        "roi_reason": row["roi_reason"],
        "priority_score": float(row["priority_score"]),
        "area_frac": float(row["area_frac"]),
        "padding_frac": float(PADDING_FRAC),
        "enc_x1": float(row["enc_x1"]),
        "enc_y1": float(row["enc_y1"]),
        "enc_x2": float(row["enc_x2"]),
        "enc_y2": float(row["enc_y2"]),
        "orig_x1": float(ox1),
        "orig_y1": float(oy1),
        "orig_x2": float(ox2),
        "orig_y2": float(oy2),
        "enc_width": int(enc_w),
        "enc_height": int(enc_h),
        "orig_width": int(orig_w),
        "orig_height": int(orig_h),
        "enc_fps": float(enc_fps),
        "orig_fps": float(orig_fps),
    })

roi_manifest_df = pd.DataFrame(roi_manifest_rows)
roi_manifest_csv = MANIFEST_DIR / "roi_manifest.csv"
roi_manifest_df.to_csv(roi_manifest_csv, index=False)

print(f"Saved ROI manifest with {len(roi_manifest_df)} rows to:")
print(roi_manifest_csv)
roi_manifest_df.head()

## 13. Sanity-check mapped boxes

In [ ]:
import matplotlib.pyplot as plt

def read_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None
    return frame

def draw_box_rgb(frame_bgr, box, color=(255, 0, 0), label=None):
    img = cv2.cvtColor(frame_bgr.copy(), cv2.COLOR_BGR2RGB)
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    if label:
        cv2.putText(img, label, (x1, max(y1 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

if len(roi_manifest_df) == 0:
    print("No ROI rows available for sanity-check visualization.")
else:
    n_show = min(4, len(roi_manifest_df))
    sample_rows = roi_manifest_df.head(n_show)

    plt.figure(figsize=(14, 4 * n_show))
    for i, (_, row) in enumerate(sample_rows.iterrows(), 1):
        enc_frame = read_frame(ENCODED_VIDEO, int(row["enc_frame_idx"]))
        orig_frame = read_frame(ORIGINAL_VIDEO, int(row["orig_frame_idx"]))

        enc_box = (row["enc_x1"], row["enc_y1"], row["enc_x2"], row["enc_y2"])
        orig_box = (row["orig_x1"], row["orig_y1"], row["orig_x2"], row["orig_y2"])

        enc_vis = draw_box_rgb(enc_frame, enc_box, label=f'{row["cls_name"]}:{row["conf"]:.2f}')
        orig_vis = draw_box_rgb(orig_frame, orig_box, label=f'{row["cls_name"]}:{row["conf"]:.2f}')

        plt.subplot(n_show, 2, 2 * i - 1)
        plt.imshow(enc_vis)
        plt.title(f'Encoded frame {int(row["enc_frame_idx"])}')
        plt.axis("off")

        plt.subplot(n_show, 2, 2 * i)
        plt.imshow(orig_vis)
        plt.title(f'Original frame {int(row["orig_frame_idx"])}')
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## 14. Save a run summary

In [ ]:
summary = {
    "sequence_name": SEQUENCE_NAME,
    "regime": REGIME,
    "original_video": str(ORIGINAL_VIDEO),
    "encoded_video": str(ENCODED_VIDEO),
    "yolo_model": YOLO_MODEL_NAME,
    "start_frame": START_FRAME,
    "max_frames": MAX_FRAMES,
    "frame_stride": FRAME_STRIDE,
    "yolo_conf": YOLO_CONF,
    "yolo_iou": YOLO_IOU,
    "min_conf_for_keep": MIN_CONF_FOR_KEEP,
    "max_rois_per_frame": MAX_ROIS_PER_FRAME,
    "small_object_area_frac": SMALL_OBJECT_AREA_FRAC,
    "padding_frac": PADDING_FRAC,
    "num_detections": int(len(detections_df)),
    "num_roi_candidates": int(len(roi_df)),
    "num_roi_manifest_rows": int(len(roi_manifest_df)),
    "detections_csv": str(detections_csv),
    "roi_candidates_csv": str(roi_csv),
    "roi_manifest_csv": str(roi_manifest_csv),
}

summary_path = MANIFEST_DIR / "run_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved summary to:")
print(summary_path)
print(json.dumps(summary, indent=2))

## 15. What this notebook completed

At this point you now have:
- YOLO detections on the encoded/base video
- filtered ROI candidates
- frame-index mapping from encoded to original video
- coordinate mapping from encoded to original video
- a clean ROI manifest for the next stage

## What Notebook 3 should do next

Notebook 3 should read `roi_manifest.csv` and:
1. reconstruct actual crops from the original and encoded videos
2. compress original crops as still images
3. measure ROI payload size
4. optionally decode compressed stills
5. prepare inputs for downstream classification and comparison